## Toolset

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import majordome as mj
from IPython.display import Markdown, display

import methane_air as ma

## Setup

In [ ]:
# Total flow rate [Nm³/h].
qdot_fuel = 500.0

# Molar composition of fuel [-].
X_fuel = "CH4: 1"

# Molar composition of oxidizer [-].
X_oxid = "N2: 0.79, O2: 0.21"

# Mechanism for base calculations.
mech = "chemkin/BFER_methane.yaml"
phase = "CH4_BFER_multi"

# Burner diameters [mm].
diam_fuel     = 27
diam_oxid_ext = 342
diam_oxid_int = 76
thick_wall    = 2.5

# Set to 1 to use pre-heated air:
hot_air = 0

# Evaluate power supply:
supply = ma.get_power_supply(mech, qdot_fuel, X_fuel, X_oxid, phase)

# Compute equivalent dimensions for a single injection:
D_fuel, D_oxid = ma.get_equivalent_diameters(
    diam_fuel, diam_oxid_ext, diam_oxid_int, thick_wall
)

# Convert transport coefficients to Sutherland:
sutherland = ma.fit_transport(mech, phase)

# Evaluate fuel conditions:
table_fuel = ma.flow_workflow(
    name = "FUEL",
    mdot = supply.fuel_mass,
    rho0 = supply.fuel_normal_density,
    T    = 298.15,
    A    = np.pi * (D_fuel / 2)**2
)

# Evaluate oxidizer conditions:
table_oxid = ma.flow_workflow(
    name = "OXID",
    mdot = supply.oxidizer_mass,
    rho0 = supply.oxidizer_normal_density,
    T    = 298.15 + hot_air * 300,
    A    = np.pi * (D_oxid / 2)**2
)

In [ ]:
#| echo: false
#| tbl-cap: Summary of energy input conditions.
#| label: tbl-summary-combustion
_ = supply.report(notebook=True)

In [ ]:
#| echo: false
#| tbl-cap: Sutherland fitting coefficients for transport properties.
#| label: tbl-sutherland-coefs
sutherland.coefs_table

In [ ]:
#| echo: false
#| tbl-cap: Conditions evaluated for fuel.
#| label: tbl-conditions-fuel
display(Markdown(table_fuel))

In [ ]:
#| echo: false
#| tbl-cap: Conditions evaluated for oxidizer.
#| label: tbl-conditions-oxidizer
display(Markdown(table_oxid))

## Postprocessing

### Cold air

In [ ]:
#| echo: false
#| fig-align: center
#| fig-cap: Temperature field for reference case.
#| label: fig-field-temperature-1
mesh = ma.load_slice()
ma.plot_slice(mesh, scalars="T", cmap="hot")

In [ ]:
#| echo: false
#| fig-align: center
#| fig-cap: Velocity magnitude field for reference case.
#| label: fig-field-velocity-1
ma.plot_slice(mesh, scalars="U", cmap="jet")

In [ ]:
ma.plot_slice(mesh, scalars="a", cmap="hot")

In [ ]:
ma.plot_slice(mesh, scalars="O2", cmap="hot")

In [ ]:
_ = ma.plot_reports(root=".")

### Pre-heated air